In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [2]:
import copy
import numpy as np
import torch

from tokenizers import Tokenizer

from storyforge.config import load_config
from storyforge.batching import get_batch
from storyforge.model import ForgeGPT

In [3]:
config = load_config("configs/default.toml")
settings = config["overfit"]

torch.manual_seed(config["seed"])
rng = np.random.default_rng(config["seed"])

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)

In [5]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

overfit_config = copy.deepcopy(config)
overfit_config["model"]["dropout"] = 0.0

In [6]:
model = ForgeGPT(
    overfit_config,
    vocab_size=tokenizer.get_vocab_size(),
).to(device)

In [7]:
print("Device:", device)

Device: mps


In [9]:
inputs, targets = get_batch(
    tokens=train_tokens,
    batch_size=settings["batch_size"],
    context=settings["context"],
    rng=rng,
    device=device,
)

model.eval()

ForgeGPT(
  (token): Embedding(8192, 384)
  (position): Embedding(256, 384)
  (drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (out_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1536, out_features=384, bias=True)
        (3): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
)

In [10]:
with torch.no_grad():
    logits, initial_loss = model(inputs, targets)

In [11]:
print("Inputs:", inputs.shape)
print("Logits:", logits.shape)
print("Initial loss:", initial_loss.item())

Inputs: torch.Size([2, 32])
Logits: torch.Size([2, 32, 8192])
Initial loss: 8.975927352905273


In [12]:
assert torch.isfinite(logits).all()
assert torch.isfinite(initial_loss)

In [13]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=settings["learning_rate"],
    weight_decay=settings["weight_decay"],
)

model.train()

ForgeGPT(
  (token): Embedding(8192, 384)
  (position): Embedding(256, 384)
  (drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (out_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1536, out_features=384, bias=True)
        (3): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
)

In [14]:
for step in range(1, settings["steps"] + 1):
    optimizer.zero_grad(set_to_none=True)

    logits, loss = model(inputs, targets)

    if not torch.isfinite(loss):
        raise RuntimeError("Training produced a non-finite loss.")

    loss.backward()
    optimizer.step()

    loss_value = loss.item()

    if step == 1 or step % 25 == 0:
        print(f"Step {step}: loss = {loss_value:.4f}")

    if loss_value < settings["target_loss"]:
        print(f"Reached the target loss at step {step}.")
        break

Step 1: loss = 8.9759
Step 25: loss = 0.3478
Reached the target loss at step 48.


In [15]:
model.eval()

with torch.no_grad():
    _, final_loss = model(inputs, targets)

print("Initial loss:", initial_loss.item())
print("Final loss:", final_loss.item())

Initial loss: 8.975927352905273
Final loss: 0.01847265660762787
